# 🤖 Baseline Accident Risk Model

In this notebook, we train a baseline model to predict whether a road segment is "Risky" (has had at least one accident) based on its physical, spatial, and traffic characteristics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_recall_curve

sns.set_theme(style='whitegrid')

# 1. Load the preprocessed dataset
DATA_PATH = '../../data/processed/features/model_dataset.parquet'
df = pd.read_parquet(DATA_PATH)

print(f"Loaded {len(df):,} segments.")

## 1. Feature Selection
We select features that describe the road's permanent character and average traffic behavior. We avoid using specific accident counts (like `acc_fatal`) as features to prevent data leakage.

In [ ]:
features = [
    # Road Geometry
    'highway_rank', 'lanes', 'length_m', 'lanes_known',
    
    # Traffic (Probe)
    'probe_count', 'speed_mean', 'has_probe_data', 
    'speed_mean_morning_peak', 'speed_mean_daytime', 'speed_mean_evening_peak',
    'pct_below_20kmh_morning_peak', 'pct_below_20kmh_daytime', 'pct_below_20kmh_evening_peak',
    
    # Spatial Context
    'dist_intersection_m', 'poi_count_200m', 'building_density_200m',
    'dist_school_m', 'dist_hospital_m', 'dist_fuel_m', 'dist_mall_m'
]

target = 'is_risky'

X = df[features]
y = df[target]

print(f"Using {len(features)} features for prediction.")

## 2. Train/Test Split
Note: In a real GIS project, we should use a **Spatial Split** (e.g., training on one district and testing on another) to ensure the model generalizes across space. For this baseline, we'll start with a stratified random split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train):,}")
print(f"Testing set size:  {len(X_test):,}")
print(f"Risk Rate in Train: {y_train.mean()*100:.1f}%")

## 3. Model Training (XGBoost)
We use `scale_pos_weight` to handle the class imbalance (more safe roads than risky roads).

In [ ]:
# Calculate class imbalance ratio
ratio = (y == 0).sum() / (y == 1).sum()

model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=ratio, # Give more importance to the risky segments
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train)
print("✅ Model trained successfully!")

## 4. Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Safe', 'Risky'], yticklabels=['Safe', 'Risky'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 5. Feature Importance
Which factors contribute most to high-risk predictions?

In [ ]:
feat_importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
feat_importances.tail(15).plot(kind='barh', color='teal')
plt.title('Top 15 Most Important Features for Risk Prediction')
plt.xlabel('F-Score (Relative Importance)')
plt.show()

## 6. Save the Model
We save the model so it can be loaded later by the Gemini-based explanation API.

In [ ]:
import pickle
import os

MODEL_DIR = '../../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

with open(os.path.join(MODEL_DIR, 'baseline_xgb_v1.pkl'), 'wb') as f:
    pickle.dump(model, f)

print(f"🚀 Model saved to {MODEL_DIR}baseline_xgb_v1.pkl")